# DAM-DRUG — Generate All Manuscript Figures

This notebook reproduces every figure in the manuscript and supplementary material.  
All outputs are written to `results/figures/`.

**Two sections:**
- **§1 Regenerate** — runs the figure script locally (requires synced CSVs/PDBs, no HPC needed)
- **§2 Display pre-generated** — shows figures whose source data lives on TRUBA (h5ad, MD traces)

Pre-requisite: sync result CSVs from TRUBA once before running §1:
```bash
rsync -av truba:$DAM_DRUG_DIR/results/phase2/GRN/regulon_auc_by_state_aggregated.csv results/phase2/GRN/
rsync -av truba:$DAM_DRUG_DIR/results/phase2/GRN/regulon_pseudotime_corr.csv        results/phase2/GRN/
rsync -av truba:$DAM_DRUG_DIR/results/phase2/LR/cellchat/                           results/phase2/LR/cellchat/
rsync -av truba:$DAM_DRUG_DIR/results/phase4/md/                                    results/phase4/md/
rsync -av truba:$DAM_DRUG_DIR/results/phase5/                                       results/phase5/
```

In [ ]:
import importlib.util
import sys
from pathlib import Path
from IPython.display import Image, display, Markdown

PROJECT  = Path().resolve()  # notebook lives at project root
FIG_DIR  = PROJECT / "results/figures"
CODE_DIR = PROJECT / "code/phase6_figures"

def run_figure(script_name: str):
    """Import a figure script by filename and call its main()."""
    path = CODE_DIR / script_name
    spec = importlib.util.spec_from_file_location("_fig", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.main()

def show(png_name: str, width: int = 900):
    """Display a figure from results/figures/."""
    p = FIG_DIR / png_name
    if p.exists():
        display(Image(str(p), width=width))
    else:
        display(Markdown(f"⚠️ `{png_name}` not found — rsync from TRUBA first."))

print(f"Project root : {PROJECT}")
print(f"Figures dir  : {FIG_DIR}")

---
## §1  Regenerate locally
Each cell below runs the figure script and then displays the output.

### Figure 2 — Target Prioritization (DGE bubble · AUCell heatmap · Pseudotime correlation)

In [ ]:
run_figure("02_fig2_targets.py")
show("fig2_target_prioritization.png")

### Figure 3 — Virtual Screening & Docking

In [ ]:
run_figure("03_fig3_docking.py")
show("fig3_docking.png")

### Figure 4 — Multi-Modal IKZF1 Validation

In [ ]:
run_figure("04_fig4_validation.py")
show("fig4_validation.png")

### Figure — CellChat L-R Interaction

In [ ]:
run_figure("05_fig_cellchat.py")
show("fig_cellchat.png")

### Graphical Abstract

In [ ]:
run_figure("06_graphical_abstract.py")
show("graphical_abstract.png")

### Supp Figure S1 — Full 46-Regulon AUCell Heatmap

In [ ]:
run_figure("07_supp_fig_s1_regulon_heatmap.py")
show("supp_fig_S1_regulon_heatmap.png")

### Supp Figure S2 — CellOracle TF Perturbation

In [ ]:
run_figure("08_supp_fig_s2_celloracle.py")
show("supp_fig_S2_celloracle.png")

### Supp Figure S6 — AlphaFold2 pLDDT Quality

In [ ]:
run_figure("11_supp_fig_s6_af2_quality.py")
show("supp_fig_S6_af2_quality.png")

### Supp Figure S8 — SLIT2/ROBO2 Expression

In [ ]:
run_figure("13_supp_fig_s8_slit2_robo2_expr.py")
show("supp_fig_S8_slit2_robo2_expr.png")

---
## §2  Display pre-generated (TRUBA data required to regenerate)

These figures require either `microglia_trajectory.h5ad` (~3 GB, TRUBA) or MD simulation
`.xvg` traces. Pre-generated PNGs are shown below.  
To regenerate, run the corresponding script on TRUBA:
```bash
apptainer exec containers/scenic.sif python code/phase6_figures/01_fig1_atlas.py
apptainer exec containers/scenic.sif python code/phase6_figures/09_supp_fig_s3_qc.py
apptainer exec containers/scenic.sif python code/phase6_figures/10_supp_fig_s5_md_rmsd.py
apptainer exec containers/scenic.sif python code/phase6_figures/12_supp_fig_s7_bhlhe_coexpr.py
```

### Figure 1 — Microglial Atlas & Substate Landscape

In [ ]:
show("fig1_atlas.png")

### Supp Figure S3 — QC & Dataset Overview

In [ ]:
show("supp_fig_S3_qc.png")

### Supp Figure S5 — MD Simulation RMSD Traces

In [ ]:
show("supp_fig_S5_md_rmsd.png")

### Supp Figure S7 — BHLHE40/41 Co-expression Rescue

In [ ]:
show("supp_fig_S7_bhlhe_coexpr.png")

---
## §3  Revised figures — enlarged fonts (reviewer response)

Reviewer: *"the font size in figures is too small … increase … by 2-3 fold."*

`code/phase6_figures/99_revise_all_figures.py` re-runs every figure script with
matplotlib monkeypatched (fonts ×2.0, tables ×1.6, panel gaps ×1.7, canvas ×1.15;
`graphical_abstract` ×1.35). Output → `results/revised_figures/`; originals in
`results/figures/` are left untouched. See `results/revised_figures/README.md`.

Figures 01 / 09 / 10 / 12 / 14 need TRUBA data — run the same driver there.


In [ ]:
import subprocess, sys

# regenerate the locally-runnable revised figures
subprocess.run(
    [sys.executable, str(CODE_DIR / "99_revise_all_figures.py"),
     "02", "03", "04", "05", "06", "07", "08", "11", "13"],
    cwd=str(PROJECT), check=True,
)

REV_DIR = FIG_DIR.parent / "revised_figures"
for png in sorted(REV_DIR.glob("*.png")):
    display(Markdown(f"**{png.name}**"))
    display(Image(str(png), width=1000))
